In [3]:
class LLaVAMemoryEstimator:
    def __init__(self, precision="fp16"):
        # LLaVA-1.5-7b Architecture Constants (from config.json)
        self.hidden_size = 4096
        self.num_hidden_layers = 32
        self.num_attention_heads = 32
        self.intermediate_size = 11008
        self.vocab_size = 32064  # Updated: 32000 -> 32064 (includes image/pad tokens)
        
        # Vision Tower (CLIP ViT-L/14-336px)
        self.vision_hidden_size = 1024
        self.vision_intermediate = 4096
        self.vision_layers = 24
        self.image_token_count = 576
        
        # Bytes per param
        self.bytes_per_param = 2 if precision in ["fp16", "bf16"] else 4
        self.precision = precision

# --- ADD THIS METHOD ---
    def format_size(self, size_bytes):
        for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
            if size_bytes < 1024.0:
                return f"{size_bytes:.2f} {unit}"
            size_bytes /= 1024.0
        return f"{size_bytes:.2f} PB"
    # -----------------------

    def estimate(self, batch_size, text_input_len, max_new_tokens):
        # -------------------------------------------
        # 1. Static Memory: Model Weights
        # -------------------------------------------
        # Precise calculation based on provided JSON configs:
        
        # A. LLM Weights (~6.74B params)
        # Embeddings + Heads (vocab dependent) + Layers (32x) + Norms
        llm_embeddings = self.vocab_size * self.hidden_size * 2 # In & Out embeddings
        llm_layers = 32 * (
            (4 * self.hidden_size * self.hidden_size) +   # Self Attn (q,k,v,o)
            (3 * self.hidden_size * self.intermediate_size) + # MLP (gate, up, down)
            (2 * self.hidden_size) # LayerNorms
        )
        llm_params = llm_embeddings + llm_layers + self.hidden_size # + Final Norm
        
        # B. Vision Tower Weights (~304M params)
        # 24 Layers of ViT-L
        vision_params = 304_000_000 

        # C. Projector Weights (~21M params)
        # MLP: Linear(1024->4096) -> GELU -> Linear(4096->4096)
        projector_params = (self.vision_hidden_size * self.hidden_size) + (self.hidden_size * self.hidden_size)

        # Total Model Memory
        total_params = llm_params + vision_params + projector_params
        model_weights_mem = total_params * self.bytes_per_param

        # -------------------------------------------
        # 2. Sequence Length Logic
        # -------------------------------------------
        total_input_tokens = text_input_len + self.image_token_count
        total_seq_len = total_input_tokens + max_new_tokens

        # -------------------------------------------
        # 3. Dynamic Memory: KV Cache
        # -------------------------------------------
        # Formula: 2 * Layers * Hidden * Batch * Seq_Len * Bytes
        kv_cache_mem = (2 * self.num_hidden_layers * self.hidden_size * batch_size * total_seq_len * self.bytes_per_param)

        # -------------------------------------------
        # 4. Dynamic Memory: Activations (Scratchpad)
        # -------------------------------------------
        # Scratchpad size for the largest operation (MLP Up-Projection)
        activation_scratchpad = (batch_size * total_input_tokens * self.intermediate_size * self.bytes_per_param)

        # Total Calculation
        total_allocated = model_weights_mem + kv_cache_mem + activation_scratchpad

        print(f"--- Corrected Estimation for Batch Size: {batch_size} ---")
        print(f"Seq Len: {total_seq_len} (Img: 576 + Text: {text_input_len} + Gen: {max_new_tokens})")
        print(f"1. Model Weights:      {self.format_size(model_weights_mem)} (Params: {total_params/1e9:.2f}B)")
        print(f"2. KV Cache:           {self.format_size(kv_cache_mem)}")
        print(f"3. Activations (Temp): {self.format_size(activation_scratchpad)}")
        print(f"-------------------------------------------")
        print(f"Predicted Allocated:   {self.format_size(total_allocated)}")
        print(f"-------------------------------------------\n")

estimator=LLaVAMemoryEstimator(precision="fp16")

In [4]:
estimator.estimate(batch_size=2, text_input_len=30, max_new_tokens=500)

--- Corrected Estimation for Batch Size: 2 ---
Seq Len: 1106 (Img: 576 + Text: 30 + Gen: 500)
1. Model Weights:      13.16 GB (Params: 7.06B)
2. KV Cache:           1.08 GB
3. Activations (Temp): 25.45 MB
-------------------------------------------
Predicted Allocated:   14.26 GB
-------------------------------------------

